In [1]:
from agents_training_facility import agents
from dotenv import load_dotenv
import os
from openai import AzureOpenAI
import json
from prompts.prompts import  plan_next_step_system, plan_next_step_user, agent_choose_tool_user
from tools import brain, common, file_handler, programer, utils_handler
from memory.memory_manager import Memory
load_dotenv()

True

In [2]:
### Utils
#-----------------------------------------------------------------------------------
def get_key( diction):
    return list(diction.keys())[0]
def get_value( diction):
    return list(diction.values())[0]

In [3]:
### Agents setup and definition 
#-----------------------------------------------------------------------------------
long_history = Memory(is_structred=False)
step_history = Memory(is_structred=True)
requests_history = Memory(is_structred=False)

manager = agents.CommandCentre('manager', ['brain', 'common'],'Brain of the operation that is planning each step -- can ask user for additional data and react to failure')
python_developer = agents.Agent('pythondeveloper', ['programer', 'common'],'main focus on writing proper python code and its execution -- suitable for most analysis and modifications')
secretary = agents.Agent('secretary', ['file_handler', 'common'],'handle everything connected with reading and writing files and directory managment')
intern = agents.Agent('intern', ['utils_handler', 'common'],' is general purpose agent, who can take most of requests that are not handled by specialists')

agents_dict = {manager.name:manager,
               python_developer.name:python_developer,
               secretary.name:secretary,
               intern.name:intern}

agents_list = manager._get_agents_characteristics()

In [4]:
#--------------------------------------------------------------------------------

user_request = input("Hello user, I am hear to assist. Please tell how can I help you?: ")
requests_history.extend_memory(user_request)


### Get info about what was done and plan what is next step and who will execute it
plan_next_step_user = plan_next_step_user.format(original_request=user_request, execution_step_history=step_history.recall_last_actions(10), avaliable_agents=agents_list)
next_step = manager._ask_agent(plan_next_step_system, plan_next_step_user, True)
step_history.extend_memory(next_step)
print(step_history.recall_last_actions(10))

if next_step.keys() == 'END' and next_step.values() == 'END':
    print('The task has been finalized, no further steps to be executed')
    #break
elif next_step.keys() == 'END' and next_step.values() == 'NO RESOURCES TO SOLVE PROBLEM':
    user_request = input("Seems like at this moment I dont have proper tools to finish this step. Could you please provide different instruction?: ")
    requests_history.extend_memory(user_request)

print(f"{user_request}           {next_step}")

secretary -- Check if titanic.csv exists
open titanic.csv           {'secretary': 'Check if titanic.csv exists'}


In [5]:
if get_key(next_step) not in agents_dict.keys():
    print(f'Agent  {get_key(next_step)} not recognized -- routing back to manager')
    step_history.extend_memory({'Last planing step failed due to incorrect agent choice -- please repeat planning step.'})
    #continue


active_agent = agents_dict[get_key(next_step)]
agent_instruction = get_value(next_step)

agent_choose_tool_user = agent_choose_tool_user.format(manager_instruction=agent_instruction, execution_step_history=step_history.recall_last_actions(10), original_request=user_request)
agent_response_dict = active_agent.think(agent_choose_tool_user)
step_history.extend_memory({f"Agent {get_key(next_step)}" : f" used tool {agent_response_dict['tool_choice']} used along with{agent_response_dict['tool_input']} arguments"})
agent_execution_result = active_agent.execute_tool(agent_response_dict)
step_history.extend_memory({'Actions result':agent_execution_result})

check_if_file_exists chosen to solve this step with titanic.csv as input


In [6]:
parse_message = f"tool {agent_response_dict['tool_choice']} used along with{agent_response_dict['tool_input']} arguments"
{f"Agent {get_key(next_step)}" : f" used tool {agent_response_dict['tool_choice']} used along with {agent_response_dict['tool_input']} arguments"}

{'Agent secretary': ' used tool check_if_file_exists used along with titanic.csv arguments'}

In [9]:
print(step_history.recall_last_actions(10))

secretary -- Check if titanic.csv exists
Agent secretary --  used tool check_if_file_exists used along withtitanic.csv arguments
Actions result -- File 'titanic.csv' not found in the current directory or any child directories.


In [11]:
import glob
import os

def search_for_files_with_given_pattern(pattern):
    """
    List all files matching python glob pattern in the current directory and all subdirectories.
    Use to understand what files are available to be chosen.

    Parameters:
    pattern (str): The python glob pattern file pattern to search for (e.g., '*.txt' for all text files, "A*" to look for all files starting with capital A).

    Returns:
    List of file paths matching the pattern in the directory and its subdirectories.
    """
    main_directory = os.path.dirname(os.path.abspath(__file__))
    
    os.chdir(main_directory)

    search_pattern = os.path.join('O:/AIAgents/aiagent/multi_ai_agent/', '**', pattern)
    matching_files = glob.glob(search_pattern, recursive=True)
    return matching_files

In [12]:
search_for_files_with_given_pattern('*joke*')

['joke.txt', 'data\\jokes.txt']